In [66]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

# check disease name 2 underwriting

In [75]:
df_dise_2_conc = pd.read_csv("./utils/disease_2_conclusion_v2.csv",keep_default_na=False)
df_dise_2_keyword = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)
# df_dise_2_keyword_kl = df_dise_2_keyword[df_dise_2_keyword["KL_match_disease"].apply(lambda x:x!="")]
df_dise_2_keyword_kl = df_dise_2_keyword

In [68]:
df_dise_2_keyword.head()

,def_name,HBZS_def,HBZS_did,disease_n,disease_list,kwords,CI,MI,ADB,Life,ZZ_match_disease,RZ_match_disease,MZ_match_disease,KL_match_disease
0,JiZhenL,,,肌阵挛,['肌阵挛'],,,,,,dianxian,,,
1,JiaZX_MiMXBB,JiaZXMMXBB,135,甲状腺弥漫性病变,['甲状腺弥漫性病变'],"[['甲状腺', '甲状', '甲壮腺', '甲超'], ['弥漫性', '非均质性改变',...",甲状腺恶性肿瘤（含原位癌）及其复发和转移,甲状腺疾病,,甲状腺恶性肿瘤（含原位癌）及其复发和转移,jiazhuangxianzhongliu,liangxingjiazhuangxianjibing,,
2,JiaZX_QieCSH,,,甲状腺切除术后,['甲状腺切除术后'],,,,,,,liangxingjiazhuangxianjibing,,
3,ZhuiTWX_ZHZ,,,椎体外系综合征,['椎体外系综合征'],,,,,,,,,
4,NongDX_NaoB,,,脓毒症相关性脑病,['脓毒症相关性脑病'],,,ruxian,,,,,,


In [76]:
#构建疾病及其相关关键词map, df来自中再疾病函数对照表
disease_n_2_list = {}
for idx,row in df_dise_2_keyword_kl.iterrows():
    disease_n = row["disease_n"].strip()
    disease_list = eval(row["disease_list"])
    # print(type(disease_list))
    try:
        if disease_n  not in disease_n_2_list and disease_n!="":
            disease_n_2_list[disease_n] = []
        disease_n_2_list[disease_n].extend(disease_list)
    except Exception as es:
        print(es)




''
''
''
''
''


In [59]:
def clear_text(text):
    text = text.strip()
    text = text.replace("\n","")
    text = text.replace("*","")
    text = text.replace("\t","")
    text = text.replace("2","II")
    text = text.replace("1","I")
    return text

disease_ns = disease_n_2_list.keys()
disease_names = list(set(df_dise_2_conc["disease_name"].tolist()))


disease_ns = [clear_text(k) for k in disease_ns]
disease_names = [clear_text(k) for k in disease_names]

print("disease_ns",len(disease_ns))
print("disease_names",len(disease_names))

disease_ns 768
disease_names 227


In [60]:
intersection = set(disease_ns) & set(disease_names)
print(len(intersection))
disease_names_only = set(disease_names) - set(disease_ns)
print(f"disease_name only have:{len(disease_names_only)}")
disease_ns_only = set(disease_ns) - set(disease_names)
print("disease_ns only have:",len(disease_ns_only))

116
disease_name only have:111
disease_ns only have: 652


In [61]:
dict_map_diff_k = {}
for k_i in disease_ns_only:
    for k_j in disease_names_only:
        if (k_i in k_j) or (k_j in k_i):
            dict_map_diff_k[k_i] = k_j

In [62]:
print(len(dict_map_diff_k))
for k,v in dict_map_diff_k.items():
    print(k,":",v)


67
肾炎 : 慢性间质性肾炎（慢性肾小管间质性肾炎）
损伤 : 脑损伤
气管炎 : 气管炎/支气管炎
肺炎 : 肺炎（不包括新冠肺炎）
静脉曲张 : （下肢）静脉曲张
高血压 : 泌尿系结石（无高血压和肾功能损害）
脑膜炎 : 脑膜炎/脑脊髓膜炎
阴道炎 : 滴虫性阴道炎
肺大疱 : 肺大疱（肺囊肿）
食管癌 : 食管
营养不良 : 肌营养不良症家族史
乳腺结节 : 乳腺结节、囊肿、占位、异常回声
子宫肉瘤 : 子宫
慢性丙型肝炎 : 丙型肝炎
子宫内膜增生 : 子宫
错构瘤 : 肾错构瘤
肝炎 : 丙型肝炎
肿瘤 : 纵膈肿瘤
胃-食管返流性疾病 : 食管
子宫内膜癌 : 子宫
先天性肾畸形 : 先天性肾畸形/单肾
肾盂肾炎 : 急性细菌性肾盂肾炎、急性肾盂肾炎
结肠炎 : 慢性结/直肠炎/溃疡性结肠炎
腮腺炎 : 流行性腮腺炎
贫血 : 继发于慢性病的贫血
急性胰腺炎 : 胰腺炎
卵巢切除手术 : 手术
急性肾小球肾炎 : 急性肾病综合征（急性肾小球肾炎、感染后肾小球肾炎）
急性丙型肝炎 : 丙型肝炎
疱疹 : 带状疱疹
子宫萎缩 : 子宫
多囊肾 : 多囊肾/家族史
子宫肥大 : 子宫
硬化性胆管炎 : 胆管炎
子宫切除手术 : 手术
慢性胰腺炎 : 胰腺炎
肠炎 : 慢性结/直肠炎/溃疡性结肠炎
囊肿 : 肺大疱（肺囊肿）
子宫腺肌病 : 子宫
溶血病 : 新生儿溶血病
溃疡性结肠炎 : 慢性结/直肠炎/溃疡性结肠炎
心包积液 : 心包炎/心包积液
喉炎 : 咽喉炎
食管良性肿瘤 : 食管
支气管扩张症 : 支气管扩张
心肌病 : 肥厚性心肌病、限制性心肌病、非致密化心肌病、致心律失常性心肌病等
肾囊肿 : 肾囊肿（排除了肾功能损害的）
子宫内膜非典型增生 : 子宫
颅脑损伤 : 脑损伤
癌 : 宫颈癌
粟粒性结核病 : 结核病
紫癜 : 特发性血小板减少紫癜
胆管结石 : 肝内胆管结石
突发耳聋 : 耳聋
心包炎 : 心包炎/心包积液
乳腺炎 : 乳腺炎、脓肿
卵圆孔未闭 : 新生儿卵圆孔未闭
气胸 : 自发性气胸
子宫炎症 : 子宫
巴氏食管 : 食管
子宫内膜腺癌 : 子宫
肾病综合征 : 肾病综合征（慢性）
心肌损害 : 新生儿心肌损害
儿童乳腺发育 : 儿童乳腺
食管炎 : 食管
肺囊肿 : 肺大疱（肺囊肿）
肺结节病 : 肺结节


In [63]:
print("disease_ns_only")
for k in disease_ns_only:
    if k not in dict_map_diff_k.keys():
        print(k)

disease_ns_only
白细胞增多
肠息肉
宫颈囊肿
前列腺结石
运动员心脏综合症
窦房传导阻滞
P波增宽
鼻腺体肥大
肺类癌
湿疹
心室肥大
门静脉高压
动脉硬化性视网膜病变
视网膜损伤
动脉硬化
病态窦房结综合征
葡萄糖尿
艾滋病
脑供血不足
角膜移植
多囊卵巢综合征
胆囊壁间结晶
卵巢细胞瘤
肠扭转
肺或支气管良性肿瘤
嗜铬细胞瘤
宫颈息肉
破伤风
骨恶性淋巴瘤
主动脉瓣关闭不全
麻疹
结节性硬化
格林巴利综合征
糖尿病性视网膜病变
虹膜炎
预激综合征
脑膜瘤
盆腔炎性疾病
先天性肾上腺增生
肺心病
外周动脉疾病
疲劳综合征
经前期综合征
主动脉瘤
ST段改变
腱鞘炎
窦性停搏
可疑
右位心
鼻外伤
室颤
室性期前收缩
库欣综合征
甲状腺切除术后
色盲
阑尾炎
阿斯珀格综合征
胆管癌
新生儿房间隔缺损
黑色素瘤
多发性硬化症家族史
冠状动脉痉挛
睾丸囊肿
骨髓抑制
心脏移植
睡眠障碍
肝恶性肿瘤
既往拒保
早产儿
神经系统肿瘤
肾盂癌
高尿酸血症
鼻咽癌
吸收不良综合征
心肌梗塞
霍乱
先天性白内障
视网膜变性
血小板减少性紫癜
青睫综合症
脑恶性肿瘤
早期复极
肾结石
多囊肾家族史
感冒
高同型半胱氨酸血症
外阴恶性肿瘤
脆性X综合征
睾丸炎
荨麻疹
存在不清晰的医疗图片
肾上腺皮质功能减退
睫状体炎
既往延期
骨软骨瘤
异食癖
输尿管结石
工作相关的肢体疾病
黑色素痣
斜视
人格障碍
虹膜睫状体炎
白血病
心肌缺血
肝脓肿
颈椎管狭窄
淋病
多发性骨髓瘤
妊娠
高胆固醇血症
膀胱癌
幽门狭窄
胆道感染
屈光不正
淋巴癌
再生障碍性贫血
胆囊切除
鼻出血
皮肤病
肝功能不全
心肌酶谱异常
沙眼
角膜疾病
尘肺
隐匿性肾炎综合征
甲状腺腺瘤
空腹血糖受损
肾功能受损
乳腺痛
前列腺囊肿
败血症
结核性胸膜炎
缺血缺氧性脑病
III三体综合征
脊髓灰质炎
胆囊肌腺症
动脉栓塞
孤独症
甲状腺功能亢进
鞘膜积液
二尖瓣反流
死亡
腹泻
视力低下
颈部肿物
滋养细胞肿瘤
发作性睡病
三尖瓣下移畸形
二叶主动脉瓣
硬膜下出血
胃术后
病毒感染
胃良性肿瘤
甲状旁腺功能亢进
库肯勃瘤
先天性胆道闭锁
终末期肾衰竭
脓毒症相关性脑病
结膜炎
甲状腺滤泡增生
胆囊疾病
低血糖
偏盲
疟疾
糖尿病家族史
小三阳
透析
癌症家族史
T波改变
房性期前收缩
十二指肠和小肠

In [64]:
print("disease_names_only")
for k in disease_names_only:
    if k  not in dict_map_diff_k.values():
        print(k)

disease_names_only
腺样体肥大
过敏性紫癜
附件
新生儿房/室间隔缺损
慢性肾小球肾炎
急性失血性贫血
腺瘤、息肉
病理分类： 乳头状癌、滤泡性癌（55岁以下）
酒精性肝病
淋菌性阴道炎
病理分类：间变性癌、未分化癌、髓样癌等
病理分类： 乳头状癌、滤泡性癌（55岁以上）
椎管狭窄已手术
失眠症
肺脓肿
二尖瓣/三尖瓣
腰椎间盘突出
胆囊炎
克罗恩病
新生儿高胆红素血症
肺错构瘤
慢性肾盂肾炎
主动脉瓣疾病
焦虑症
新生儿缺氧缺血性脑病
声带息肉
肠吸收不良
甲亢
肾积水
胆结石
子宫内膜增厚（内膜厚度超过I0mm）
脾脏创伤
扩张性心肌病
视网膜血管病变
各类肾炎，包含其他因素
肛周脓肿
急性间质性肾炎（急性感染性肾小管间质性肾炎）
阻塞性睡眠呼吸暂停综合征
糖尿病前期
视网膜血管变性
霉菌性阴道炎
颈椎椎管狭窄
泌尿系感染
面神经炎（面瘫）
心脏肥大/增大/扩大
巨幼细胞性贫血
皮肌炎
HP阳性、幽门螺杆菌感染
色素性视网膜炎
甲状腺手术
肺动脉瓣疾病
疑似错构瘤的肺结节
盆腔积液
美尼尔氏病
血压升高
四肢骨折
其他心脏结构异常
•急性轴索型神经病•急性特发性炎症性多发性神经病•急性炎症性脱髓鞘性多发性神经病 (AIDP)•急性炎症性多发性神经病 (AIPN)•急性运动性轴索型神经病 (AMAN)•急性运动感觉性轴索型神经病 (AMSAN)•慢性炎症性脱髓鞘性神经病 (CIDP)•慢性炎症性多发性神经病 (CIP)•慢性复发性炎症性多发性神经病 (CIRP)•多发性神经根性神经炎•吉兰-巴雷综合征•格林-巴利综合征•GB综合征（Guillain-Barré）
腰椎椎管狭窄
视网膜脱离
男性乳腺
脑梗塞
宫颈纳囊
纵膈气肿
心律失常
甲减


In [29]:
k

' 地中海贫血'

In [8]:
df_dise_2_conc.head()

,disease_name,file_name,conclusion
0,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
1,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
2,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
3,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
4,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."


In [2]:
#大模型
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-7B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

# 编辑距离计算疾病相似度

In [3]:
def edit_distance(str1, str2):
    """计算两个字符串的编辑距离

    Args:
        str1: 字符串1
        str2: 字符串2

    Returns:
        int: 编辑距离
    """

    m = len(str1)
    n = len(str2)

    # 初始化二维数组dp，dp[i][j]表示str1[:i]和str2[:j]的编辑距离
    dp = [[i+j for j in range(n+1)] for i in range(m+1)]
    for i in range(1, m+1):
        dp[i][0] = i
    for j in range(1, n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)

    return dp[m][n]

def normalized_similarity(str1, str2):
    distance = edit_distance(str1, str2)
    max_len = max(len(str1), len(str2))
    similarity = 1 - distance / max_len
    return similarity

# 示例用法
str1 = "kitten"
# str2 = "sitting"
str2 = "kitte"

distance = edit_distance(str2, str1)
print("编辑距离:", distance)

distance_nor = normalized_similarity(str1,str2)
print(round(distance_nor,4))


编辑距离: 1
0.8333


In [10]:
# 计算疾病相似度，并返回top_n最相近的疾病
def disease_similarity(input_dise,disease_n_2_list,top_n=3):
    disease_similarity_score = {}
    for k_dise,v_diseKey in disease_n_2_list.items():
        disease_similarity_score[k_dise] = max([normalized_similarity(input_dise,keyw) for keyw in v_diseKey])

    sorted_dict = sorted(disease_similarity_score.items(), key=lambda x: x[1], reverse=True)
    result =sorted_dict[:top_n]
    return result
    

# 读取 中再疾病函数对照表的disease_n 到 disease_2_concluson中的disease_name的 map
# 读取 中再疾病函数对照表的disease_n 到 key word 的map

In [7]:
with open("./utils/KL_disease_n_2_name_map.json","r",encoding="utf-8") as f:
    map_key_2_dise = json.load(f)


with open("./utils/KL_disease_n_2_keyword.json","r",encoding="utf-8")as f:
    disease_n_2_list = json.load(f)

In [6]:
map_key_2_dise["肾结石"]

[['胆结石', 0.6666666666666667],
 ['肝内胆管结石', 0.33333333333333337],
 ['肾积水', 0.33333333333333337],
 ['肺结节', 0.33333333333333337],
 ['肾错构瘤', 0.25]]

In [69]:
#测试样本
df = pd.read_csv("./规则引擎对比结果-评点数据.csv",keep_default_na=False)

#昆仑核保结论知识
df_dise_2_conc = pd.read_csv("./disease_2_conclusion.csv",keep_default_na=False)  

In [77]:
df.columns

Index(['姓名', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果'],
      dtype='object')

In [74]:
for idx,row in df.iterrows():
    url = row["线上页面与结果"].split("\n")
    print(url)
    break

['URL http://yc-dev.smart-insight-service.com:41373/underwrite/Image_underwriting/detail.html?id=78472&risk_case_id=87234', '重疾 延期至术后或痊愈后', '医疗 延期至术后或痊愈后']


In [70]:
df.head()

,姓名,编号（身份证号）,性别\n（1:男，2:女，0:未知）,医院,日期,临床诊断（化验项、疾病等以下划线拼接）,账单金额（数字）,年龄,编号（案件编号）,图片名/文件名,attach id,图片分类/票据类别,影像报告内容image_report,data_source：类别 如昆仑体检告知,线上页面与结果
0,张颖,421023199012060444,2,市到大学深圳医院,,"宫腔内稍高回声团,考虑子宫内膜息肉",,34,M202444111993763,ile_TWzoqlM3_20241119142423324.jpg,,,"{'report_name': '', 'des_dic': '经阴道三维超声检查：\\n后...",体检,URL http://yc-dev.smart-insight-service.com:41...
1,马春兰,330621199503154709,2,,,子宫肌瘤可能,,29,M202431111993911,file_4BqWBucI_20241119194907173.jpg,,,"{'report_name': '', 'des_dic': '【子宫】 【经阴道】\\n...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
2,高刚正,440303198910301317,1,,,双肾结石(沙粒样),,35,M202444111893402,file_Oj01H8zK_20241118225440966.jpg,,,"{'report_name': '', 'des_dic': '双肾轮廓清晰。 切面形态大...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
3,高刚正,440303198910301317,1,,,脂肪肝（中度）,,35,M202444111893402,file_Oj01H8zK_20241118225440966.jpg,,,"{'report_name': '', 'des_dic': '肝脏切面轮廊清晰， 右叶斜...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
4,郑序华,440582199008156187,2,,,双肾结石,,34,M202444111893285,file_n3wYhN9Y_20241118142225174.jpg,,,"{'report_name': '', 'des_dic': '双肾形态大小位置正常 包膜...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...


In [34]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_time_use = []
for idx,row in df.iterrows():
    
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    print("病人基本信息",basic_info)
    print(image_report)
    # if image_report != "":
    #     image_report = json.loads(image_report)
    #     print(type(image_report))
    #     query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    # else:
    #     query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    query = diagnose
    
    
    try:
        s_time = time.time()
        #获得相似疾病name
        map_disease_name= disease_similarity(query,disease_n_2_list)
        max_simi_dise_names = map_key_2_dise.get(map_disease_name[0][0],[map_disease_name[0]])
        #获得相似疾病的结论
        recall_kbs = [ df_dise_2_conc[df_dise_2_conc["disease_name"]==max_simi_dis[0]]["conclusion"].tolist() for max_simi_dis in max_simi_dise_names]
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)
        ans_list = [recall_kb for recall_kb in recall_kbs if len(recall_kb)>0]
        ans_list = ans_list[0]

    except Exception as es:
        print(es)
        ans_list = []

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,output_struct=output_struct)
    # print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)

    results_underwriting.append(result)
    recall_query.append(ans_list)

    

病人基本信息 年龄:34,性别:女性,临床诊断:宫腔内稍高回声团,考虑子宫内膜息肉,影像报告:{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
recall time use: 0.1831493377685547
病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm

In [35]:
sum(recall_time_use)/len(recall_time_use)

0.09962441772222519

In [36]:
df["RAG核保结论"] = results_underwriting
df["recall_query"] = recall_query
df.to_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv",index=False)

In [37]:
len_get_recall = len([rec_res for rec_res in recall_query if len(rec_res)!=0])
print(len_get_recall)
print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")

32
召回率：100.0%


In [38]:
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_FullTextSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_SemanticSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_HybridSearch.csv")
# df_full = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv")


# recall_query = df_full["recall_query"].tolist()
# len_get_recall = len([rec_res for rec_res in recall_query if len(eval(rec_res))!=0])
# print(len_get_recall)
# print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")
bad_dise = []
for idx,row in df.iterrows():
    if len(row["recall_query"]) == 0:
        bad_dise.append(row["临床诊断（化验项、疾病等以下划线拼接）"])
print(len(bad_dise))
print(bad_dise)

0
[]


In [25]:
df_dise_2_conc[df_dise_2_conc["file_name"]=="子宫.txt"].head(30)

,disease_name,file_name,conclusion
812,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
813,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
814,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
815,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '非功能失调性子宫出血', '资料': '病历、妇科超..."
816,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '绝经后子宫出血', '资料': '病历、妇科超声、血..."
817,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '待进行子宫切除', '资料': '病历、妇科超声、血..."
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."


In [33]:
query_dise = bad_dise[1]
print(query_dise)
results_query = disease_similarity(query_dise,disease_n_2_list)
print(results_query)
print(results_query[0][0])
map_disease_name = map_key_2_dise.get(results_query[0][0],[results_query[0]])
print(map_disease_name)
df_dise_2_conc[df_dise_2_conc["disease_name"]==map_disease_name].head()

子宫肌瘤可能
[('子宫肌瘤', 0.6666666666666667), ('子宫肉瘤', 0.5), ('子宫切除手术', 0.33333333333333337)]
子宫肌瘤
子宫肌瘤


,disease_name,file_name,conclusion
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
